# AdaptiveMath-AI — Baseline models

**Notebook 03 of 06**

Simple, psychometric, educational and classical machine-learning baselines are fitted on the common training partition. Candidate selection uses validation ROC-AUC only.

## 1. Environment, metrics, and immutable model inputs

Establish a common evaluation function and prevent accidental use of current-response or protected identifier fields. Load the scalar modeling Parquet, verify split counts, define the full metric suite at a fixed 0.5 threshold, and select only safe engineered numeric predictors.

Leakage control: `is_correct`, identifiers, raw context IDs, current answer choice, current confidence, and `CorrectAnswer` are excluded.

In [1]:
from pathlib import Path
import json, math, os, time, textwrap, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display
import psutil, joblib
from scipy.special import expit, logit
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,f1_score,precision_score,recall_score,roc_auc_score,average_precision_score,matthews_corrcoef,cohen_kappa_score,log_loss,brier_score_loss,confusion_matrix)

mpl.rcParams.update({"font.family":"Times New Roman","font.size":18,"axes.titlesize":20,"axes.labelsize":19,"xtick.labelsize":18,"ytick.labelsize":18,"legend.fontsize":18,"figure.titlesize":20,"axes.spines.top":False,"axes.spines.right":False,"savefig.dpi":350,"figure.dpi":350})
PALETTE={"blue":"#2F6B8F","orange":"#C77C2B","green":"#4F7F3A","purple":"#6D5A8D","red":"#B45A55","gray":"#6E7378","light_gray":"#E7EAED","teal":"#4A8C8A","gold":"#C9A227","black":"#222222","white":"#FFFFFF"}
SEED=20260713; np.random.seed(SEED)
cwd=Path.cwd().resolve(); ROOT=next((p for p in [cwd,*cwd.parents] if (p/"notebooks").is_dir() and (p/"README.md").exists()),None)
if ROOT is None: raise FileNotFoundError("Run from the repository root or the notebooks directory.")
MODEL_DIR=ROOT/"models/baseline_model_artifacts"; MODEL_DIR.mkdir(parents=True,exist_ok=True)
data=pd.read_parquet(ROOT/"dataset/processed/modeling_dataset.parquet")
forbidden={"answer_value","Confidence","correct_answer","missing_confidence","history_eligible"}
assert forbidden.isdisjoint(data.columns), f"Forbidden prediction-time fields present: {forbidden & set(data.columns)}"
splits={s:data[data.split_id==s].copy() for s in ["train","validation","temporal_test","external_unseen_student","external_unseen_question"]}
excluded={"interaction_id","answer_id","user_id","question_id","date_answered","split_id","is_correct","group_id","quiz_id","scheme_of_work_id","session_id","primary_subject_name","academic_term","primary_subject_id","parent_subject_id"}
numeric_features=[c for c in data.columns if c not in excluded and pd.api.types.is_numeric_dtype(data[c])]
# Protected attributes are retained for robustness but excluded from core baselines.
numeric_features=[c for c in numeric_features if c not in {"gender","premium_pupil"}]
X={s:g[numeric_features] for s,g in splits.items()}; y={s:g.is_correct.astype(int).to_numpy() for s,g in splits.items()}
display(pd.DataFrame({"split":list(splits),"rows":[len(splits[s]) for s in splits],"correct_rate":[y[s].mean() for s in splits]}))
print(f"Safe numeric predictors: {len(numeric_features)}")
print(numeric_features)

def save_and_show_figure(fig,filename):
 path=ROOT/"figures"/filename; fig.savefig(path,dpi=350,bbox_inches="tight",facecolor="white"); plt.show(); print(f"Saved figure: {path}"); return path
def annotate_bars(ax,fmt="{:.3f}"):
 for p in ax.patches: ax.text(p.get_x()+p.get_width(),p.get_y()+p.get_height()/2," "+fmt.format(p.get_width()),va="center",clip_on=False)
def wrap_labels(labels,width=28): return ["\n".join(textwrap.wrap(str(x),width)) for x in labels]
def plot_horizontal_metric_ranking(frame,label_col,metric_col,color=PALETTE["blue"]):
 d=frame.sort_values(metric_col); fig,ax=plt.subplots(figsize=(13,max(6,.72*len(d)))); ax.barh(wrap_labels(d[label_col]),d[metric_col],color=color); annotate_bars(ax); return fig,ax
def place_legend_below(ax,ncol=3): ax.legend(loc="upper center",bbox_to_anchor=(.5,-.16),ncol=ncol,frameon=False)

def safe_probability(p): return np.clip(np.asarray(p,dtype=float),1e-6,1-1e-6)
def calibration_errors(y_true,p,n_bins=15):
 p=safe_probability(p); edges=np.linspace(0,1,n_bins+1); ids=np.clip(np.digitize(p,edges,right=True)-1,0,n_bins-1); ece=0.; mce=0.
 for b in range(n_bins):
  mask=ids==b
  if mask.any():
   gap=abs(float(y_true[mask].mean())-float(p[mask].mean())); ece+=mask.mean()*gap; mce=max(mce,gap)
 return ece,mce
def metric_row(model_name,y_true,p,split_id):
 p=safe_probability(p); pred=(p>=.5).astype(int); tn,fp,fn,tp=confusion_matrix(y_true,pred,labels=[0,1]).ravel(); ece,mce=calibration_errors(y_true,p)
 try: auc=roc_auc_score(y_true,p); pr=average_precision_score(y_true,p)
 except Exception: auc=pr=np.nan
 try:
  z=logit(p).reshape(-1,1); cal=LogisticRegression(C=1e6,solver="lbfgs",max_iter=200).fit(z,y_true); slope=float(cal.coef_[0,0]); intercept=float(cal.intercept_[0])
 except Exception: slope=intercept=np.nan
 return {"Model":model_name,"Split":split_id,"N":len(y_true),"Accuracy":accuracy_score(y_true,pred),"Balanced_Accuracy":balanced_accuracy_score(y_true,pred),"Macro_F1":f1_score(y_true,pred,average="macro",zero_division=0),"Weighted_F1":f1_score(y_true,pred,average="weighted",zero_division=0),"Precision":precision_score(y_true,pred,zero_division=0),"Recall":recall_score(y_true,pred,zero_division=0),"Specificity":tn/max(tn+fp,1),"ROC_AUC":auc,"PR_AUC":pr,"MCC":matthews_corrcoef(y_true,pred),"Kappa":cohen_kappa_score(y_true,pred),"Log_Loss":log_loss(y_true,p,labels=[0,1]),"Brier_Score":brier_score_loss(y_true,p),"ECE":ece,"MCE":mce,"FPR":fp/max(fp+tn,1),"FNR":fn/max(fn+tp,1),"Calibration_Slope":slope,"Calibration_Intercept":intercept,"Confusion_Matrix":f"[[{tn},{fp}],[{fn},{tp}]]"}
print("Metric contract includes discrimination, threshold, calibration, and confusion-matrix evidence.")

,split,rows,correct_rate
0,train,281459,0.651711
1,validation,60750,0.634700
2,temporal_test,63209,0.613267
3,external_unseen_student,22687,0.659144
4,external_unseen_question,21452,0.633181


Safe numeric predictors: 59
['active_learning_days', 'age_at_answer', 'bkt_mastery', 'collaborative_state_norm', 'day_of_week', 'days_since_previous_interaction', 'days_since_previous_subject_interaction', 'elo_probability', 'elo_question_difficulty', 'elo_student_ability', 'ewm_accuracy', 'historical_group_performance', 'historical_quiz_performance', 'historical_scheme_performance', 'hour', 'inactivity_gap', 'interaction_position', 'matrix_factorization_score', 'missing_context_metadata', 'missing_demographics', 'missing_student_metadata', 'month', 'new_session', 'number_of_groups', 'number_of_quizzes', 'number_of_schemes', 'number_of_subjects', 'prior_answer_choice_1_rate', 'prior_answer_choice_2_rate', 'prior_answer_choice_3_rate', 'prior_answer_choice_4_rate', 'prior_confidence_mean', 'prior_correct_count', 'prior_cumulative_accuracy', 'prior_incorrect_count', 'prior_interaction_count', 'prior_subject_accuracy', 'prior_subject_correct', 'prior_subject_opportunities', 'question_cold

## 2. Simple and historical baselines

Establish transparent lower bounds based on class prevalence, learner history, item difficulty, subject difficulty, and backoff. Fit constants/maps from `train`; use prior-only learner history already emitted by Notebook 02.

Leakage control: Validation/test labels do not enter any baseline estimate; the student mean is shifted and question/subject mappings are train-only.

In [2]:
train_mean=float(y["train"].mean())
def simple_predictions(frame):
 prior_n=frame.prior_interaction_count.to_numpy(float); prior_correct=frame.prior_correct_count.to_numpy(float); q=frame.question_historical_accuracy.to_numpy(float)
 return {
  "Majority Class":np.full(len(frame),.999 if train_mean>=.5 else .001),
  "Global Correctness Mean":np.full(len(frame),train_mean),
  "Student Historical Mean":frame.prior_cumulative_accuracy.to_numpy(float),
  "Question Historical Mean":q,
  "Student-Question Backoff Mean":(prior_correct+20*q)/(prior_n+20),
  "Subject Historical Mean":frame.subject_historical_accuracy.to_numpy(float),
 }
all_predictions={s:{} for s in splits}
for s,frame in splits.items(): all_predictions[s].update(simple_predictions(frame))
simple_artifact={"train_correctness_mean":train_mean,"threshold":.5,"definitions":list(simple_predictions(splits["validation"]).keys())}
p=MODEL_DIR/"simple_baselines.json"; p.write_text(json.dumps(simple_artifact,indent=2),encoding="utf-8")
print(f"Saved model artifact: {p} ({p.stat().st_size/1024:.1f} KiB)")
preview=pd.DataFrame({k:v[:5] for k,v in simple_predictions(splits["validation"]).items()}); display(preview)

Saved model artifact: <repository_root>/models/baseline_model_artifacts/simple_baselines.json (0.3 KiB)


,Majority Class,Global Correctness Mean,Student Historical Mean,Question Historical Mean,Student-Question Backoff Mean,Subject Historical Mean
0,0.999,0.651711,0.900000,0.651711,0.829060,0.647684
1,0.999,0.651711,0.882353,0.651711,0.817383,0.647684
2,0.999,0.651711,0.865385,0.668296,0.810638,0.647684
3,0.999,0.651711,0.849057,0.725318,0.815156,0.711601
4,0.999,0.651711,0.851852,0.679794,0.805350,0.711601


## 3. Psychometric and educational baselines

Compare mechanism-oriented educational models with generic classifiers. Evaluate explicitly named Rasch-style, 2PL-inspired, fixed-parameter BKT, PFA-style, AFM-style, and exposure-SVD proxies alongside Elo. These compact comparators test mechanisms; they are not claimed as canonical reimplementations. Their fitted calibration layers use training only.

Leakage control: Psychometric states were emitted before current outcomes; all fitted coefficients use `train` labels only.

In [3]:
# Direct prior-state models.
for s,frame in splits.items():
 all_predictions[s]["Rasch-style logit baseline"]=expit(frame.rasch_ability.to_numpy()-frame.rasch_question_difficulty.to_numpy())
 all_predictions[s]["Elo Rating"]=frame.elo_probability.to_numpy(float)
 all_predictions[s]["Fixed-parameter BKT proxy"]=frame.bkt_mastery.to_numpy(float)
# Smoothed empirical discrimination for a stable 2PL approximation.
train_disc=splits["train"].groupby("question_id").apply(lambda g:g.is_correct.corr(g.rasch_ability) if len(g)>=30 and g.is_correct.nunique()>1 else np.nan,include_groups=False).fillna(0)
discrimination=(1+2*train_disc.abs()).clip(.5,2.5)
for s,frame in splits.items():
 a=frame.question_id.map(discrimination).fillna(1).to_numpy(); theta_minus_b=frame.rasch_ability.to_numpy()-frame.rasch_question_difficulty.to_numpy(); all_predictions[s]["2PL-inspired smoothed baseline"]=expit(a*theta_minus_b)
# PFA: counts of prior successes and failures at the concept level.
pfa_features=["prior_subject_correct","prior_subject_opportunities"]
pfa_X={s:pd.DataFrame({"success":splits[s].prior_subject_correct,"failure":splits[s].prior_subject_opportunities-splits[s].prior_subject_correct}) for s in splits}
pfa=Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler()),("model",LogisticRegression(C=1.0,max_iter=500,random_state=SEED))]); pfa.fit(pfa_X["train"],y["train"])
for s in splits: all_predictions[s]["PFA-style logistic baseline"]=pfa.predict_proba(pfa_X[s])[:,1]
# AFM: concept intercepts plus opportunity count.
afm_pre=ColumnTransformer([("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),["prior_subject_opportunities"]),("concept",OneHotEncoder(handle_unknown="ignore",min_frequency=10),["primary_subject_name"])])
afm=Pipeline([("pre",afm_pre),("model",LogisticRegression(C=.5,max_iter=500,random_state=SEED))]); afm.fit(splits["train"][["prior_subject_opportunities","primary_subject_name"]],y["train"])
for s in splits: all_predictions[s]["AFM-style logistic baseline"]=afm.predict_proba(splits[s][["prior_subject_opportunities","primary_subject_name"]])[:,1]
# Train-exposure matrix factorization features receive a train-only logistic observation model.
mf_features=["student_question_similarity","collaborative_state_norm","matrix_factorization_score"]
mf=Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler()),("model",LogisticRegression(C=1,max_iter=500,random_state=SEED))]); mf.fit(splits["train"][mf_features],y["train"])
for s in splits: all_predictions[s]["Exposure-SVD collaborative proxy"]=mf.predict_proba(splits[s][mf_features])[:,1]
for name,model in {"pfa.joblib":pfa,"afm.joblib":afm,"matrix_factorization.joblib":mf}.items():
 p=MODEL_DIR/name; joblib.dump(model,p); print(f"Saved model: {p} ({p.stat().st_size/1024:.1f} KiB)")
irt_path=MODEL_DIR/"irt_discrimination.parquet"; discrimination.rename("discrimination").reset_index().to_parquet(irt_path,index=False); print(f"Saved model parameters: {irt_path} ({irt_path.stat().st_size/1024:.1f} KiB)")
display(pd.DataFrame({"educational_model":["Rasch-style logit baseline","2PL-inspired smoothed baseline","Elo Rating","Fixed-parameter BKT proxy","PFA-style logistic baseline","AFM-style logistic baseline","Exposure-SVD collaborative proxy"],"canonical_reimplementation":[False,False,True,False,False,False,False],"training_labels_used":[False,True,False,False,True,True,True],"validation_labels_used":[False]*7}))

Saved model: <repository_root>/models/baseline_model_artifacts/pfa.joblib (1.9 KiB)
Saved model: <repository_root>/models/baseline_model_artifacts/afm.joblib (16.0 KiB)
Saved model: <repository_root>/models/baseline_model_artifacts/matrix_factorization.joblib (2.0 KiB)
Saved model parameters: <repository_root>/models/baseline_model_artifacts/irt_discrimination.parquet (165.8 KiB)


,educational_model,canonical_reimplementation,training_labels_used,validation_labels_used
0,Rasch-style logit baseline,False,False,False
1,2PL-inspired smoothed baseline,False,True,False
2,Elo Rating,True,False,False
3,Fixed-parameter BKT proxy,False,False,False
4,PFA-style logistic baseline,False,True,False
5,AFM-style logistic baseline,False,True,False
6,Exposure-SVD collaborative proxy,False,True,False


## 4. Classical machine-learning candidates and validation-only selection

Establish strong tabular baselines under a common feature and tuning protocol. Compare two pre-declared configurations for logistic regression, Gaussian Naive Bayes, decision tree, random forest, Extra Trees, histogram gradient boosting, and MLP; select each family by validation ROC-AUC.

Leakage control: Fitting uses `train` only; candidate ranking uses `validation` only. External and temporal-test labels remain unopened until configurations are frozen.

In [4]:
impute=SimpleImputer(strategy="median",add_indicator=True)
def pipe(est,scale=False):
 steps=[("imputer",SimpleImputer(strategy="median",add_indicator=True))]
 if scale: steps.append(("scale",StandardScaler()))
 steps.append(("model",est)); return Pipeline(steps)
candidates={
 "Logistic Regression":[pipe(LogisticRegression(C=.1,max_iter=500,random_state=SEED),True),pipe(LogisticRegression(C=1,max_iter=500,random_state=SEED),True)],
 "Naive Bayes":[pipe(GaussianNB(var_smoothing=1e-9)),pipe(GaussianNB(var_smoothing=1e-7))],
 "Decision Tree":[pipe(DecisionTreeClassifier(max_depth=8,min_samples_leaf=40,class_weight="balanced",random_state=SEED)),pipe(DecisionTreeClassifier(max_depth=14,min_samples_leaf=20,class_weight="balanced",random_state=SEED))],
 "Random Forest":[pipe(RandomForestClassifier(n_estimators=70,max_depth=14,min_samples_leaf=15,max_features="sqrt",n_jobs=-1,class_weight="balanced_subsample",random_state=SEED)),pipe(RandomForestClassifier(n_estimators=90,max_depth=20,min_samples_leaf=10,max_features=.5,n_jobs=-1,class_weight="balanced_subsample",random_state=SEED))],
 "Extra Trees":[pipe(ExtraTreesClassifier(n_estimators=70,max_depth=16,min_samples_leaf=12,max_features="sqrt",n_jobs=-1,class_weight="balanced",random_state=SEED)),pipe(ExtraTreesClassifier(n_estimators=90,max_depth=None,min_samples_leaf=15,max_features=.5,n_jobs=-1,class_weight="balanced",random_state=SEED))],
 "HistGradientBoosting":[pipe(HistGradientBoostingClassifier(max_iter=140,learning_rate=.05,max_leaf_nodes=31,l2_regularization=1,random_state=SEED)),pipe(HistGradientBoostingClassifier(max_iter=100,learning_rate=.1,max_leaf_nodes=31,l2_regularization=2,random_state=SEED))],
 "MLPClassifier":[pipe(MLPClassifier(hidden_layer_sizes=(32,),alpha=1e-3,batch_size=1024,max_iter=100,early_stopping=True,validation_fraction=.1,n_iter_no_change=10,tol=1e-4,random_state=SEED),True),pipe(MLPClassifier(hidden_layer_sizes=(64,32),alpha=1e-3,batch_size=1024,max_iter=100,early_stopping=True,validation_fraction=.1,n_iter_no_change=10,tol=1e-4,random_state=SEED),True)],
}
selection_rows=[]; trained={}; efficiency=[]
for family,models in candidates.items():
 family_results=[]
 for idx,model in enumerate(models):
  start=time.perf_counter(); model.fit(X["train"],y["train"]); train_seconds=time.perf_counter()-start
  start=time.perf_counter(); pval=model.predict_proba(X["validation"])[:,1]; val_seconds=time.perf_counter()-start
  auc=roc_auc_score(y["validation"],pval); ll=log_loss(y["validation"],safe_probability(pval)); family_results.append((auc,-ll,idx,model,train_seconds,val_seconds))
  selection_rows.append({"Model_Family":family,"Candidate":idx+1,"Validation_ROC_AUC":auc,"Validation_Log_Loss":ll,"Parameters":json.dumps(model.get_params(),default=str)[:2000]})
 family_results.sort(key=lambda z:(z[0],z[1]),reverse=True); auc,negll,idx,best,train_seconds,val_seconds=family_results[0]; trained[family]=best
 artifact=MODEL_DIR/(family.lower().replace(" ","_")+".joblib"); joblib.dump(best,artifact)
 total_eval=pd.concat([X[s] for s in ["validation","temporal_test","external_unseen_student","external_unseen_question"]],ignore_index=True)
 start=time.perf_counter(); _=best.predict_proba(total_eval)[:,1]; inference=time.perf_counter()-start
 param_count=0
 if family=="MLPClassifier": param_count=int(sum(w.size for w in best.named_steps["model"].coefs_)+sum(b.size for b in best.named_steps["model"].intercepts_))
 efficiency.append({"Model":family,"Training_Time_Seconds":train_seconds,"Inference_Time_Seconds":inference,"Evaluated_Interactions":len(total_eval),"Latency_Milliseconds_Per_Interaction":1000*inference/len(total_eval),"Throughput_Interactions_Per_Second":len(total_eval)/max(inference,1e-9),"Model_Size_Bytes":artifact.stat().st_size,"Parameter_Count":param_count,"Peak_Process_RSS_Bytes":psutil.Process().memory_info().rss})
 print(f"Selected {family} candidate {idx+1}: validation ROC-AUC={auc:.4f}; saved {artifact} ({artifact.stat().st_size/2**20:.2f} MiB)")
selection=pd.DataFrame(selection_rows); selected_lookup={k:int(selection[(selection.Model_Family==k)].sort_values(["Validation_ROC_AUC","Validation_Log_Loss"],ascending=[False,True]).iloc[0].Candidate) for k in candidates}
selection["Selected"]=selection.apply(lambda r:r.Candidate==selected_lookup[r.Model_Family],axis=1)
selection.to_csv(ROOT/"artifacts/baseline_hyperparameter_summary.csv",index=False)
for family,model in trained.items():
 for s in splits: all_predictions[s][family]=model.predict_proba(X[s])[:,1]
efficiency=pd.DataFrame(efficiency); efficiency.to_csv(ROOT/"artifacts/baseline_training_efficiency.csv",index=False)
display(selection[selection.Selected][["Model_Family","Candidate","Validation_ROC_AUC","Validation_Log_Loss"]].sort_values("Validation_ROC_AUC",ascending=False))
display(efficiency.sort_values("Training_Time_Seconds"))
print(f"Saved table: {ROOT/'artifacts/baseline_hyperparameter_summary.csv'}")
print(f"Saved table: {ROOT/'artifacts/baseline_training_efficiency.csv'}")
print("XGBoost, LightGBM, CatBoost, and a dedicated factorization-machine package were not installed in the pre-existing environment; no substitute is mislabeled as those algorithms.")

Selected Logistic Regression candidate 1: validation ROC-AUC=0.7677; saved <repository_root>/models/baseline_model_artifacts/logistic_regression.joblib (0.01 MiB)


Selected Naive Bayes candidate 1: validation ROC-AUC=0.7297; saved <repository_root>/models/baseline_model_artifacts/naive_bayes.joblib (0.01 MiB)


Selected Decision Tree candidate 1: validation ROC-AUC=0.7604; saved <repository_root>/models/baseline_model_artifacts/decision_tree.joblib (0.04 MiB)


Selected Random Forest candidate 1: validation ROC-AUC=0.7711; saved <repository_root>/models/baseline_model_artifacts/random_forest.joblib (36.39 MiB)


Selected Extra Trees candidate 2: validation ROC-AUC=0.7700; saved <repository_root>/models/baseline_model_artifacts/extra_trees.joblib (182.23 MiB)


Selected HistGradientBoosting candidate 2: validation ROC-AUC=0.7743; saved <repository_root>/models/baseline_model_artifacts/histgradientboosting.joblib (0.43 MiB)


Selected MLPClassifier candidate 1: validation ROC-AUC=0.7693; saved <repository_root>/models/baseline_model_artifacts/mlpclassifier.joblib (0.06 MiB)


,Model_Family,Candidate,Validation_ROC_AUC,Validation_Log_Loss
11,HistGradientBoosting,2,0.774307,0.542327
6,Random Forest,1,0.771120,0.568600
9,Extra Trees,2,0.770044,0.568632
12,MLPClassifier,1,0.769325,0.548193
0,Logistic Regression,1,0.767750,0.551271
4,Decision Tree,1,0.760420,0.584817
2,Naive Bayes,1,0.729702,2.517798


,Model,Training_Time_Seconds,Inference_Time_Seconds,Evaluated_Interactions,Latency_Milliseconds_Per_Interaction,Throughput_Interactions_Per_Second,Model_Size_Bytes,Parameter_Count,Peak_Process_RSS_Bytes
1,Naive Bayes,1.424152,0.135294,168098,0.000805,1.242465e+06,5553,0,1040007168
0,Logistic Regression,2.625732,0.092045,168098,0.000548,1.826259e+06,5985,0,1137770496
2,Decision Tree,5.449231,0.082247,168098,0.000489,2.043821e+06,42419,0,792788992
5,HistGradientBoosting,5.888296,0.253891,168098,0.001510,6.620875e+05,451425,0,1173127168
3,Random Forest,7.870307,0.302429,168098,0.001799,5.558266e+05,38156882,0,940654592
6,MLPClassifier,10.627784,0.198606,168098,0.001181,8.463911e+05,63649,2081,511311872
4,Extra Trees,14.197621,0.449640,168098,0.002675,3.738503e+05,191077666,0,859537408


Saved table: <repository_root>/artifacts/baseline_hyperparameter_summary.csv
Saved table: <repository_root>/artifacts/baseline_training_efficiency.csv
XGBoost, LightGBM, CatBoost, and a dedicated factorization-machine package were not installed in the pre-existing environment; no substitute is mislabeled as those algorithms.


## 5. Frozen evaluation, predictions, and short-history stress test

Compare every baseline under future-interaction and entity cold-start conditions after validation-only selection. Compute the full metric contract, save row-level probabilities with identifiers, and evaluate the external unseen-student target immediately after exactly 5, 10, 20, or 50 prior interactions.

Leakage control: Test and external results do not alter models, hyperparameters, feature lists, or the 0.5 threshold.

In [5]:
metric_tables={}; prediction_tables={}
file_map={"validation":"validation","temporal_test":"test","external_unseen_student":"external_unseen_student","external_unseen_question":"external_unseen_question"}
for split_id,suffix in file_map.items():
 rows=[]; frames=[]; base=splits[split_id][["interaction_id","answer_id","user_id","question_id","is_correct","split_id","prior_interaction_count"]].reset_index(drop=True)
 for model_name,pred in all_predictions[split_id].items():
  rows.append(metric_row(model_name,y[split_id],pred,split_id)); tmp=base.copy(); tmp["Model"]=model_name; tmp["Predicted_Probability"]=safe_probability(pred); frames.append(tmp)
 metrics=pd.DataFrame(rows).sort_values(["ROC_AUC","Log_Loss"],ascending=[False,True]); predictions=pd.concat(frames,ignore_index=True)
 metric_tables[split_id]=metrics; prediction_tables[split_id]=predictions
 metrics_path=ROOT/"artifacts"/f"baseline_{suffix}_metrics.csv"; pred_path=ROOT/"artifacts"/f"baseline_predictions_{suffix}.parquet"
 metrics.to_csv(metrics_path,index=False); predictions.to_parquet(pred_path,index=False,compression="zstd")
 print(f"Saved table: {metrics_path}"); print(f"Saved predictions: {pred_path} ({pred_path.stat().st_size/2**20:.1f} MiB)")
 display(metrics[["Model","N","ROC_AUC","Log_Loss","Brier_Score","ECE","Accuracy","Macro_F1"]].head(12))
# External unseen-student stress targets are a subset of the frozen external predictions.
stress=pd.read_parquet(ROOT/"dataset/processed/stress_short_history.parquet")
stress_rows=[]
for cap,g in stress.groupby("history_cap"):
 for model_name,preds in all_predictions["external_unseen_student"].items():
  lookup=dict(zip(splits["external_unseen_student"].answer_id,safe_probability(preds))); p=np.array([lookup[a] for a in g.answer_id]); stress_rows.append({**metric_row(model_name,g.is_correct.to_numpy(),p,"stress_short_history"),"History_Cap":int(cap)})
short_metrics=pd.DataFrame(stress_rows); short_path=ROOT/"artifacts/baseline_short_history_metrics.csv"; short_metrics.to_csv(short_path,index=False)
print(f"Saved table: {short_path}"); display(short_metrics[["History_Cap","Model","N","ROC_AUC","Log_Loss","Brier_Score"]].sort_values(["History_Cap","ROC_AUC"],ascending=[True,False]).groupby("History_Cap").head(5))

Saved table: <repository_root>/artifacts/baseline_validation_metrics.csv
Saved predictions: <repository_root>/artifacts/baseline_predictions_validation.parquet (14.0 MiB)


,Model,N,ROC_AUC,Log_Loss,Brier_Score,ECE,Accuracy,Macro_F1
18,HistGradientBoosting,60750,0.774307,0.542327,0.182806,0.010775,0.725383,0.690769
16,Random Forest,60750,0.771120,0.568600,0.193677,0.095145,0.705942,0.694974
17,Extra Trees,60750,0.770044,0.568632,0.193799,0.092873,0.703572,0.692729
19,MLPClassifier,60750,0.769325,0.548193,0.184977,0.019016,0.721811,0.685699
13,Logistic Regression,60750,0.767750,0.551271,0.185987,0.022266,0.719984,0.677385
15,Decision Tree,60750,0.760420,0.584817,0.200253,0.106561,0.690123,0.680832
7,Elo Rating,60750,0.742959,0.576386,0.196035,0.047067,0.704955,0.650862
6,Rasch-style logit baseline,60750,0.741109,0.607886,0.209053,0.114401,0.674156,0.542682
9,2PL-inspired smoothed baseline,60750,0.734846,0.635806,0.214707,0.132637,0.674156,0.542682
14,Naive Bayes,60750,0.728535,2.517798,0.296732,0.286029,0.677613,0.666125


Saved table: <repository_root>/artifacts/baseline_test_metrics.csv
Saved predictions: <repository_root>/artifacts/baseline_predictions_test.parquet (14.6 MiB)


,Model,N,ROC_AUC,Log_Loss,Brier_Score,ECE,Accuracy,Macro_F1
18,HistGradientBoosting,63209,0.774554,0.550852,0.186310,0.016224,0.718489,0.693232
16,Random Forest,63209,0.771354,0.573886,0.195635,0.089063,0.700944,0.694826
17,Extra Trees,63209,0.769923,0.574416,0.195957,0.087022,0.698524,0.692357
19,MLPClassifier,63209,0.769605,0.557038,0.188569,0.023747,0.713269,0.686757
13,Logistic Regression,63209,0.768702,0.560163,0.189499,0.028969,0.713190,0.680552
15,Decision Tree,63209,0.761399,0.590136,0.201957,0.101246,0.688873,0.683933
7,Elo Rating,63209,0.745915,0.581418,0.198365,0.044268,0.696958,0.654343
6,Rasch-style logit baseline,63209,0.740686,0.628794,0.217526,0.131047,0.658466,0.539325
9,2PL-inspired smoothed baseline,63209,0.735597,0.654184,0.222777,0.147258,0.658466,0.539325
14,Naive Bayes,63209,0.722689,2.664406,0.306880,0.296302,0.668338,0.662220


Saved table: <repository_root>/artifacts/baseline_external_unseen_student_metrics.csv
Saved predictions: <repository_root>/artifacts/baseline_predictions_external_unseen_student.parquet (4.6 MiB)


,Model,N,ROC_AUC,Log_Loss,Brier_Score,ECE,Accuracy,Macro_F1
18,HistGradientBoosting,22687,0.768982,0.535432,0.180276,0.007845,0.726010,0.676321
16,Random Forest,22687,0.766719,0.562656,0.190772,0.097392,0.709569,0.688456
19,MLPClassifier,22687,0.766022,0.538005,0.181242,0.006856,0.726187,0.674981
17,Extra Trees,22687,0.765125,0.563035,0.191273,0.096480,0.707189,0.687253
13,Logistic Regression,22687,0.764991,0.541364,0.182379,0.024744,0.723101,0.653208
15,Decision Tree,22687,0.754426,0.584828,0.200388,0.119399,0.692291,0.676975
14,Naive Bayes,22687,0.737582,2.113726,0.268207,0.256764,0.706528,0.663592
6,Rasch-style logit baseline,22687,0.735047,0.591196,0.202730,0.100028,0.683167,0.511576
7,Elo Rating,22687,0.727449,0.583233,0.198904,0.069729,0.707101,0.636191
9,2PL-inspired smoothed baseline,22687,0.727008,0.619636,0.208617,0.121406,0.683167,0.511576


Saved table: <repository_root>/artifacts/baseline_external_unseen_question_metrics.csv
Saved predictions: <repository_root>/artifacts/baseline_predictions_external_unseen_question.parquet (4.9 MiB)


,Model,N,ROC_AUC,Log_Loss,Brier_Score,ECE,Accuracy,Macro_F1
18,HistGradientBoosting,21452,0.740658,0.570610,0.194426,0.011871,0.701939,0.662148
19,MLPClassifier,21452,0.739273,0.576650,0.196343,0.031976,0.696951,0.651528
13,Logistic Regression,21452,0.737408,0.577688,0.197088,0.036200,0.696578,0.635940
16,Random Forest,21452,0.736380,0.601685,0.207440,0.107223,0.674809,0.663705
17,Extra Trees,21452,0.734972,0.599543,0.206724,0.100841,0.675461,0.663552
15,Decision Tree,21452,0.723819,0.626949,0.218385,0.129093,0.638868,0.636189
14,Naive Bayes,21452,0.717727,2.427420,0.289059,0.276669,0.687302,0.657251
7,Elo Rating,21452,0.712643,0.599679,0.206192,0.051733,0.684645,0.640664
2,Student Historical Mean,21452,0.702849,0.732964,0.209426,0.049167,0.681055,0.616171
6,Rasch-style logit baseline,21452,0.702586,0.635356,0.219982,0.117609,0.656256,0.494615


Saved table: <repository_root>/artifacts/baseline_short_history_metrics.csv


,History_Cap,Model,N,ROC_AUC,Log_Loss,Brier_Score
13,5,Logistic Regression,271,0.755348,0.522971,0.174386
16,5,Random Forest,271,0.754393,0.569459,0.194300
17,5,Extra Trees,271,0.753247,0.568885,0.194551
18,5,HistGradientBoosting,271,0.750446,0.532637,0.179481
4,5,Student-Question Backoff Mean,271,0.750127,0.555370,0.185584
39,10,MLPClassifier,271,0.781948,0.521187,0.174014
38,10,HistGradientBoosting,271,0.775898,0.523178,0.174713
36,10,Random Forest,271,0.775096,0.559305,0.189327
33,10,Logistic Regression,271,0.772997,0.532445,0.178520
37,10,Extra Trees,271,0.770836,0.561155,0.191053


## 6. Validation-selected comparators

The strongest simple, psychometric and classical models are identified from validation performance and carried forward without using temporal-test or external-holdout results for selection.

In [6]:
val=metric_tables["validation"]; families={
 "best_simple":val[val.Model.isin(["Majority Class","Global Correctness Mean","Student Historical Mean","Question Historical Mean","Student-Question Backoff Mean","Subject Historical Mean"])],
 "best_psychometric":val[val.Model.isin(["Rasch-style logit baseline","2PL-inspired smoothed baseline","Elo Rating","Fixed-parameter BKT proxy","PFA-style logistic baseline","AFM-style logistic baseline","Exposure-SVD collaborative proxy"])],
 "best_classical":val[val.Model.isin(list(candidates))]}
ranking=[]
for label,g in families.items():
 r=g.sort_values(["ROC_AUC","Log_Loss"],ascending=[False,True]).iloc[0]; ranking.append({"Comparator_Class":label,"Selected_Model":r.Model,"Validation_ROC_AUC":r.ROC_AUC,"Validation_Log_Loss":r.Log_Loss})
ranking=pd.DataFrame(ranking); display(ranking); display(metric_tables["temporal_test"][["Model","ROC_AUC","Log_Loss","Brier_Score","ECE"]].head(10)); display(metric_tables["external_unseen_student"][["Model","ROC_AUC","Log_Loss","Brier_Score"]].head(10)); display(efficiency)

,Comparator_Class,Selected_Model,Validation_ROC_AUC,Validation_Log_Loss
0,best_simple,Student-Question Backoff Mean,0.722820,0.584862
1,best_psychometric,Elo Rating,0.742959,0.576386
2,best_classical,HistGradientBoosting,0.774307,0.542327


,Model,ROC_AUC,Log_Loss,Brier_Score,ECE
18,HistGradientBoosting,0.774554,0.550852,0.186310,0.016224
16,Random Forest,0.771354,0.573886,0.195635,0.089063
17,Extra Trees,0.769923,0.574416,0.195957,0.087022
19,MLPClassifier,0.769605,0.557038,0.188569,0.023747
13,Logistic Regression,0.768702,0.560163,0.189499,0.028969
15,Decision Tree,0.761399,0.590136,0.201957,0.101246
7,Elo Rating,0.745915,0.581418,0.198365,0.044268
6,Rasch-style logit baseline,0.740686,0.628794,0.217526,0.131047
9,2PL-inspired smoothed baseline,0.735597,0.654184,0.222777,0.147258
14,Naive Bayes,0.722689,2.664406,0.306880,0.296302


,Model,ROC_AUC,Log_Loss,Brier_Score
18,HistGradientBoosting,0.768982,0.535432,0.180276
16,Random Forest,0.766719,0.562656,0.190772
19,MLPClassifier,0.766022,0.538005,0.181242
17,Extra Trees,0.765125,0.563035,0.191273
13,Logistic Regression,0.764991,0.541364,0.182379
15,Decision Tree,0.754426,0.584828,0.200388
14,Naive Bayes,0.737582,2.113726,0.268207
6,Rasch-style logit baseline,0.735047,0.591196,0.202730
7,Elo Rating,0.727449,0.583233,0.198904
9,2PL-inspired smoothed baseline,0.727008,0.619636,0.208617


,Model,Training_Time_Seconds,Inference_Time_Seconds,Evaluated_Interactions,Latency_Milliseconds_Per_Interaction,Throughput_Interactions_Per_Second,Model_Size_Bytes,Parameter_Count,Peak_Process_RSS_Bytes
0,Logistic Regression,2.625732,0.092045,168098,0.000548,1.826259e+06,5985,0,1137770496
1,Naive Bayes,1.424152,0.135294,168098,0.000805,1.242465e+06,5553,0,1040007168
2,Decision Tree,5.449231,0.082247,168098,0.000489,2.043821e+06,42419,0,792788992
3,Random Forest,7.870307,0.302429,168098,0.001799,5.558266e+05,38156882,0,940654592
4,Extra Trees,14.197621,0.449640,168098,0.002675,3.738503e+05,191077666,0,859537408
5,HistGradientBoosting,5.888296,0.253891,168098,0.001510,6.620875e+05,451425,0,1173127168
6,MLPClassifier,10.627784,0.198606,168098,0.001181,8.463911e+05,63649,2081,511311872


,artifact,exists
0,artifacts/baseline_validation_metrics.csv,True
1,artifacts/baseline_test_metrics.csv,True
2,artifacts/baseline_external_unseen_student_metric...,True
3,artifacts/baseline_external_unseen_question_metri...,True
4,artifacts/baseline_short_history_metrics.csv,True
5,artifacts/baseline_predictions_validation.parquet,True
6,artifacts/baseline_predictions_test.parquet,True
7,artifacts/baseline_predictions_external_unseen_st...,True
8,artifacts/baseline_predictions_external_unseen_qu...,True
9,artifacts/baseline_training_efficiency.csv,True
